# Task 13 — Simple FastAPI to Serve Predictions

This notebook shows the FastAPI code and how to run it. The actual API is in `app.py`.

### Step 1 — Train and save the model

In [2]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../data/loan_data.csv")
df.dropna(inplace=True)
df = df.drop_duplicates().reset_index(drop=True)

le = LabelEncoder()
df['education'] = le.fit_transform(df['education'])

feature_cols = ['age','income','loan_amount','credit_score','employment_years','education']
X = df[feature_cols]
y = df['approved']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

model = LogisticRegression()
model.fit(X_train_s, y_train)

# Save model and scaler
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model and scaler saved.")

Model and scaler saved.


To run the API:

  pip install fastapi uvicorn
  uvicorn app:app --reload

Then open: http://127.0.0.1:8000/docs

Test with curl:

  curl -X POST http://127.0.0.1:8000/predict \\
    -H "Content-Type: application/json" \\
    -d '{
      "age": 35,
      "income": 60000,
      "loan_amount": 20000,
      "credit_score": 680,
      "years_employed": 8,
      "education": 0
    }'

Expected response: {"approved": true, "probability": 0.82}